# Combinar el DART de Jhoan con el V6 del compañero

Se compara la combinación sobre las 11 noches de prueba de la primera ronda, para la que se conservan los dos modelos y los vectores de la CRNN. DART usa las características tabulares. El V6 usa su CRNN, LightGBM sobre 192 características y HMM. Se fijó la misma mezcla del experimento con la CRNN pequeña antes de observar el resultado aquí. Los pesos son 75 % para DART y 25 % para V6.

La ruta de Viterbi del V6 entrega etiquetas. Para la mezcla se calculan las probabilidades marginales de etapa del mismo HMM por noche mediante el algoritmo hacia delante y hacia atrás. El control V6 reproduce la ruta de Viterbi ya evaluada. Los artefactos locales de la primera ronda proceden de la comparación justa y de su medición de tiempos. Las salidas guardadas se pueden leer sin esos archivos.

In [1]:
import json
import sys
from pathlib import Path
from tempfile import gettempdir
import numpy as np
import pandas as pd
from lightgbm import Booster
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display
ROOT = next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'.dvc/config').is_file())
sys.path.insert(0,str(ROOT/'packages/sleep-staging/src'))
sys.path.insert(0,str(ROOT))
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import VALID_STAGES, discover_sleep_telemetry_records, discover_sleep_cassette_records
from sleep_staging.training import load_or_build_supervised_dataset
from scripts.compare_cassette_hybrid import decode_night

cache = ROOT/'.sleep-cv/8f83a22b1d3a65d3'
modelos = cache/'tiempos-ronda-1'
stages = np.array(VALID_STAGES)
spec = json.loads((ROOT/'datos/experimentos/lightgbm_st_sc/experimento.json').read_text('utf8'))
orden_dart = json.loads((modelos/'mediciones.json').read_text('utf8'))['clases_dart']
dart = Booster(model_file=str(modelos/'dart.txt'))
v6 = Booster(model_file=str(modelos/'v6.txt'))
guardados = pd.concat([pd.read_parquet(cache/f'fold-{i}-dart.parquet') for i in range(1,6)],ignore_index=True)
val = guardados[guardados.fold.eq(1)].copy()
train = guardados[guardados.fold.ne(1)]
assert set(train.subject).isdisjoint(val.subject)
claves = ['record_id','epoch_index','subject','stage','fold']
otros = pd.read_parquet(cache/'fold-1-v6.parquet')
val = val.merge(otros,on=claves,validate='one_to_one',suffixes=('_dart','_v6')).sort_values(['record_id','epoch_index']).reset_index(drop=True)
assert len(val) == len(otros) == guardados.fold.eq(1).sum()
registros = set(val.record_id)
def bloque(lista, tipo, ubicacion):
    ds = load_or_build_supervised_dataset(lista,PreprocessingPipeline(PreprocessingConfig(**spec[tipo])),ubicacion,verbose=False)
    sel = ds.metadata.record_id.isin(registros)
    x = ds.features.loc[sel].astype('float32').reindex(columns=spec['feature_columns'])
    x.index = pd.MultiIndex.from_frame(ds.metadata.loc[sel,['record_id','epoch_index']])
    return x
st = discover_sleep_telemetry_records(ROOT/'datos/sleep-telemetry')
sc = discover_sleep_cassette_records(ROOT/'datos/sleep-cassette')
partes = [bloque(st,'preprocessing_telemetry',ROOT/'datos/processed/supervised')]
partes.extend(bloque([r],'preprocessing_cassette',Path(gettempdir())/'sleep-edfx-sc-features-v1') for r in sc if r.record_id in registros)
X = pd.concat(partes).loc[pd.MultiIndex.from_frame(val[['record_id','epoch_index']])]
assert X.index.is_unique and len(X)==len(val)
p_dart = dart.predict(X,num_threads=4)[:,[orden_dart.index(s) for s in stages]]
assert np.array_equal(stages[p_dart.argmax(1)],val.prediction_dart)

C:\Users\sebas\AppData\Local\Temp\ipykernel_37872\2456336010.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  X = pd.concat(partes).loc[pd.MultiIndex.from_frame(val[['record_id','epoch_index']])]


In [2]:
counts = np.ones((len(stages),len(stages)))
prior = np.ones(len(stages))
for _,g in train.groupby('record_id'):
    g = g.sort_values('epoch_index')
    c = pd.Categorical(g.stage,categories=stages).codes
    junto = np.diff(g.epoch_index)==1
    np.add.at(counts,(c[:-1][junto],c[1:][junto]),1)
    np.add.at(prior,c[np.r_[True,~junto]],1)
A = counts / counts.sum(1,keepdims=True)
prior /= prior.sum()
def marginales(prob):
    alpha = np.empty_like(prob)
    alpha[0] = prior*prob[0]
    alpha[0] /= alpha[0].sum()
    for t in range(1,len(prob)):
        alpha[t] = (alpha[t-1] @ A) * prob[t]
        alpha[t] /= alpha[t].sum()
    beta = np.ones(len(stages))
    post = alpha.copy()
    for t in range(len(prob)-2,-1,-1):
        beta = A @ (prob[t+1]*beta)
        beta /= beta.sum()
        post[t] *= beta
    return post/post.sum(axis=1,keepdims=True)

p_v6 = np.empty_like(p_dart)
for rid,g in val.groupby('record_id'):
    vectores = np.load(cache/'v6-1'/f'{rid}-192.npy',mmap_mode='r')
    prob = v6.predict(vectores,num_threads=4)
    traza = stages[decode_night(prob,A,prior)[g.epoch_index.to_numpy()]]
    np.testing.assert_array_equal(traza,g.prediction_v6)
    p_v6[g.index] = marginales(prob)[g.epoch_index.to_numpy()]
assert np.isfinite(p_v6).all() and np.allclose(p_v6.sum(1),1)
val['mezcla'] = stages[(0.75*p_dart+0.25*p_v6).argmax(1)]
val['cohorte'] = val.record_id.str[:2].map({'ST':'Telemetry','SC':'Cassette'})
filas = []
for c,g in [('Ambas',val),*val.groupby('cohorte')]:
    for etiqueta,col in [('DART','prediction_dart'),('V6 con HMM (Viterbi)','prediction_v6'),('75 % DART + 25 % V6','mezcla')]:
        por_clase = f1_score(g.stage,g[col],labels=stages,average=None,zero_division=0)
        filas.append([c,etiqueta,g.subject.nunique(),g.record_id.nunique(),len(g),por_clase.mean(),accuracy_score(g.stage,g[col]),*por_clase])
display(pd.DataFrame(filas,columns=['Cohorte','Modelo','Personas','Noches','Segmentos','F1 macro','Exactitud',*stages]).round(4))

,Cohorte,Modelo,Personas,Noches,Segmentos,F1 macro,Exactitud,W,N1,N2,N3,REM
0,Ambas,DART,7,11,15893,0.8194,0.8745,0.9757,0.6373,0.8320,0.7887,0.8631
1,Ambas,V6 con HMM (Viterbi),7,11,15893,0.7749,0.8232,0.8935,0.5369,0.8046,0.7986,0.8411
2,Ambas,75 % DART + 25 % V6,7,11,15893,0.8364,0.8893,0.9782,0.6528,0.8589,0.8156,0.8766
3,Cassette,DART,3,3,8039,0.7394,0.9167,0.9855,0.4558,0.8283,0.5633,0.8639
4,Cassette,V6 con HMM (Viterbi),3,3,8039,0.6337,0.8098,0.9045,0.4706,0.6716,0.3193,0.8026
5,Cassette,75 % DART + 25 % V6,3,3,8039,0.7633,0.9283,0.9888,0.4968,0.8553,0.5895,0.8862
6,Telemetry,DART,4,8,7854,0.8262,0.8313,0.9082,0.7037,0.8339,0.8225,0.8629
7,Telemetry,V6 con HMM (Viterbi),4,8,7854,0.7985,0.8369,0.8197,0.5600,0.8853,0.8760,0.8513
8,Telemetry,75 % DART + 25 % V6,4,8,7854,0.8393,0.8492,0.9041,0.7089,0.8607,0.8486,0.8741


In [3]:
from sklearn.metrics import confusion_matrix
rng = np.random.default_rng(42)
remuestras = []
for _,cohorte in val.groupby('cohorte'):
    matrices = np.array([[confusion_matrix(g.stage,g[c],labels=stages) for c in ['prediction_dart','mezcla']]
                         for _,g in cohorte.groupby('subject')])
    indices = rng.integers(len(matrices),size=(5000,len(matrices)))
    remuestras.append(matrices[indices].sum(axis=1))
cm = sum(remuestras)
den = cm.sum(axis=-2)+cm.sum(axis=-1)
f1 = np.divide(2*np.diagonal(cm,axis1=-2,axis2=-1),den,
               out=np.zeros(den.shape),where=den!=0).mean(axis=-1)
ic = np.quantile(f1[:,1]-f1[:,0],[0.025,0.975])
delta = f1_score(val.stage,val.mezcla,average='macro')-f1_score(val.stage,val.prediction_dart,average='macro')
print(f'Diferencia de F1 macro frente a DART: {delta:+.4f}; IC 95 % orientativo por persona [{ic[0]:+.4f}, {ic[1]:+.4f}].')

Diferencia de F1 macro frente a DART: +0.0170; IC 95 % orientativo por persona [+0.0055, +0.0266].
